# CSV + API

In this reboot, we are going to use:

- The [Goodreads books](https://www.kaggle.com/jealousleopard/goodreadsbooks) dataset from Kaggle.
- The [Open Library Books API](https://openlibrary.org/dev/docs/api/books)

The goal of this livecode is to load the data from a CSV + loop over rows to enrich each row with information such as:

- List of subjects (Science, Humor, Travel, etc.)
- The cover URL of the book
- Other information you'd find useful in the JSON API

First, download the CSV in the local folder:

In [4]:
!curl -L https://gist.githubusercontent.com/ssaunier/351b17f5a7a009808b60aeacd1f4a036/raw/books.csv > data/books.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1509k  100 1509k    0     0  12.2M      0 --:--:-- --:--:-- --:--:-- 12.3M


In [6]:

!ls -lh data/

total 4408
-rw-r--r--@ 1 gonul  staff    60K Apr 24 20:08 ML_Titanic_dataset (1).csv
-rw-r--r--@ 1 gonul  staff    15K Apr 24 20:58 U.S._crude_oil_production.csv
-rw-r--r--@ 1 gonul  staff    13K Apr 24 20:58 U.S._natural_gas_production.csv
-rw-r--r--@ 1 gonul  staff   1.5M Apr 25 08:14 books.csv


Then import the usual suspects!

In [14]:
import requests
import pandas as pd
import numpy as np

## Load books from CSV

In [9]:
# YOUR CODE HERE
df = pd.read_csv('data/books.csv')
df.head()


,bookID,title,authors,average_rating,isbn,isbn13,language_code,# num_pages,ratings_count,text_reviews_count
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling-Mary GrandPré,4.56,0439785960,9780439785969,eng,652,1944099,26249
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling-Mary GrandPré,4.49,0439358078,9780439358071,eng,870,1996446,27613
2,3,Harry Potter and the Sorcerer's Stone (Harry P...,J.K. Rowling-Mary GrandPré,4.47,0439554934,9780439554930,eng,320,5629932,70390
3,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.41,0439554896,9780439554893,eng,352,6267,272
4,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling-Mary GrandPré,4.55,043965548X,9780439655484,eng,435,2149872,33964


In [11]:
df.dtypes

bookID                  int64
title                  object
authors                object
average_rating        float64
isbn                   object
isbn13                  int64
language_code          object
# num_pages             int64
ratings_count           int64
text_reviews_count      int64
dtype: object

Let's add a new column

In [12]:
# YOUR CODE HERE
df['cover_url'] = None
df.head()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,# num_pages,ratings_count,text_reviews_count,cover_url
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling-Mary GrandPré,4.56,0439785960,9780439785969,eng,652,1944099,26249,None
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling-Mary GrandPré,4.49,0439358078,9780439358071,eng,870,1996446,27613,None
2,3,Harry Potter and the Sorcerer's Stone (Harry P...,J.K. Rowling-Mary GrandPré,4.47,0439554934,9780439554930,eng,320,5629932,70390,None
3,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.41,0439554896,9780439554893,eng,352,6267,272,None
4,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling-Mary GrandPré,4.55,043965548X,9780439655484,eng,435,2149872,33964,None


## API - Open Library

In [29]:
# YOUR CODE HERE
def fetch_book(isbn):
    url = 'https://openlibrary.org/api/books'
    params = {
        'bibkeys': f'ISBN:{isbn}',
        'format': 'json',
        'jscmd': 'data'
        }
    respone = requests.get(url, params=params)
    data = respone.json()
    if data and f'ISBN:{isbn}' in data:
        return data[f'ISBN:{isbn}']
    return
        


In [31]:
#res = fetch_book("9780439785969")
#res

## Calling the API with multiple ISBNs at a time

In [33]:
# YOUR CODE HERE
%time

for index, row in df.head(15).iterrows():
    # If the book has no cover URL, fetch it
    if row['cover_url'] is None:
        isbn = row['isbn13']
        print(f"Fetching cover for {row['title']}")
        
        book = fetch_book(isbn)
        if book:
            cover_url = book.get('cover', {}).get('large', '')
            df.loc[index, 'cover_url'] = cover_url
        else:
            df.loc[index, 'cover_url'] = ''

CPU times: user 2 μs, sys: 0 ns, total: 2 μs
Wall time: 5.25 μs
Fetching cover for Harry Potter and the Half-Blood Prince (Harry Potter  #6)
Fetching cover for Harry Potter and the Order of the Phoenix (Harry Potter  #5)
Fetching cover for Harry Potter and the Sorcerer's Stone (Harry Potter  #1)
Fetching cover for Harry Potter and the Chamber of Secrets (Harry Potter  #2)
Fetching cover for Harry Potter and the Prisoner of Azkaban (Harry Potter  #3)
Fetching cover for Harry Potter Boxed Set  Books 1-5 (Harry Potter  #1-5)
Fetching cover for Unauthorized Harry Potter Book Seven News: "Half-Blood Prince" Analysis and Speculation
Fetching cover for Harry Potter Collection (Harry Potter  #1-6)
Fetching cover for The Ultimate Hitchhiker's Guide: Five Complete Novels and One Story (Hitchhiker's Guide to the Galaxy  #1-5)
Fetching cover for The Ultimate Hitchhiker's Guide to the Galaxy
Fetching cover for The Hitchhiker's Guide to the Galaxy (Hitchhiker's Guide to the Galaxy  #1)
Fetching cove

In [34]:
df.head(15)

,bookID,title,authors,average_rating,isbn,isbn13,language_code,# num_pages,ratings_count,text_reviews_count,cover_url
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling-Mary GrandPré,4.56,0439785960,9780439785969,eng,652,1944099,26249,https://covers.openlibrary.org/b/id/15156081-L...
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling-Mary GrandPré,4.49,0439358078,9780439358071,eng,870,1996446,27613,https://covers.openlibrary.org/b/id/12025650-L...
2,3,Harry Potter and the Sorcerer's Stone (Harry P...,J.K. Rowling-Mary GrandPré,4.47,0439554934,9780439554930,eng,320,5629932,70390,https://covers.openlibrary.org/b/id/7572543-L.jpg
3,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.41,0439554896,9780439554893,eng,352,6267,272,https://covers.openlibrary.org/b/id/10301720-L...
4,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling-Mary GrandPré,4.55,043965548X,9780439655484,eng,435,2149872,33964,https://covers.openlibrary.org/b/id/8778528-L.jpg
5,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling-Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,38872,154,https://covers.openlibrary.org/b/id/278981-L.jpg
6,9,"Unauthorized Harry Potter Book Seven News: ""Ha...",W. Frederick Zimmerman,3.69,0976540606,9780976540601,en-US,152,18,1,https://covers.openlibrary.org/b/id/742235-L.jpg
7,10,Harry Potter Collection (Harry Potter #1-6),J.K. Rowling,4.73,0439827604,9780439827607,eng,3342,27410,820,https://covers.openlibrary.org/b/id/279436-L.jpg
8,12,The Ultimate Hitchhiker's Guide: Five Complete...,Douglas Adams,4.38,0517226952,9780517226957,eng,815,3602,258,https://covers.openlibrary.org/b/id/12617870-L...
9,13,The Ultimate Hitchhiker's Guide to the Galaxy,Douglas Adams,4.38,0345453743,9780345453747,eng,815,240189,3954,https://covers.openlibrary.org/b/id/14656530-L...


## Calling the API with multiple ISBNs at a time

In [35]:
isbns = [9780439785969, 9780439358071, 9780439554930]
[f"ISBN:{isbn}" for isbn in isbns]

['ISBN:9780439785969', 'ISBN:9780439358071', 'ISBN:9780439554930']

In [36]:
",".join([f"ISBN:{isbn}" for isbn in isbns])

'ISBN:9780439785969,ISBN:9780439358071,ISBN:9780439554930'

In [37]:
def fetch_books(isbns):
    # Define the URL and build bibkeys from ISBN
    url = "https://openlibrary.org/api/books"
    bibkeys = ",".join([f"ISBN:{isbn}" for isbn in isbns])
    
    # Define parameters for HTTP request
    params = {
        'bibkeys': bibkeys,
        'format': 'json',
        'jscmd': 'data'
    }
    
    # Perform request
    response = requests.get(url, params=params).json()
    
    return response

In [38]:
df.set_index("isbn13", inplace=True)
df.head()

,bookID,title,authors,average_rating,isbn,language_code,# num_pages,ratings_count,text_reviews_count,cover_url
isbn13,,,,,,,,,,
9780439785969,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling-Mary GrandPré,4.56,0439785960,eng,652,1944099,26249,https://covers.openlibrary.org/b/id/15156081-L...
9780439358071,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling-Mary GrandPré,4.49,0439358078,eng,870,1996446,27613,https://covers.openlibrary.org/b/id/12025650-L...
9780439554930,3,Harry Potter and the Sorcerer's Stone (Harry P...,J.K. Rowling-Mary GrandPré,4.47,0439554934,eng,320,5629932,70390,https://covers.openlibrary.org/b/id/7572543-L.jpg
9780439554893,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.41,0439554896,eng,352,6267,272,https://covers.openlibrary.org/b/id/10301720-L...
9780439655484,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling-Mary GrandPré,4.55,043965548X,eng,435,2149872,33964,https://covers.openlibrary.org/b/id/8778528-L.jpg


In [39]:
!pip install tqdm

In [ ]:
%time

from tqdm import tqdm

for group in tqdm(np.array_split(df.head(100), 5)): # 5 groups of 20 books
    books = fetch_books(list(group.index))
    
    for isbn_code, book in books.items():
        isbn = int(isbn_code.strip("ISBN:"))
        df.loc[isbn, "cover_url"] = book.get("cover", {}).get("large", "")

/Users/gonul/.pyenv/versions/workintech/lib/python3.12/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


CPU times: user 3 μs, sys: 3 μs, total: 6 μs
Wall time: 7.15 μs


100%|██████████| 5/5 [00:07<00:00,  1.41s/it]


In [41]:
df.head(-5)

,bookID,title,authors,average_rating,isbn,language_code,# num_pages,ratings_count,text_reviews_count,cover_url
isbn13,,,,,,,,,,
9780439785969,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling-Mary GrandPré,4.56,0439785960,eng,652,1944099,26249,https://covers.openlibrary.org/b/id/15156081-L...
9780439358071,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling-Mary GrandPré,4.49,0439358078,eng,870,1996446,27613,https://covers.openlibrary.org/b/id/12025650-L...
9780439554930,3,Harry Potter and the Sorcerer's Stone (Harry P...,J.K. Rowling-Mary GrandPré,4.47,0439554934,eng,320,5629932,70390,https://covers.openlibrary.org/b/id/7572543-L.jpg
9780439554893,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.41,0439554896,eng,352,6267,272,https://covers.openlibrary.org/b/id/10301720-L...
9780439655484,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling-Mary GrandPré,4.55,043965548X,eng,435,2149872,33964,https://covers.openlibrary.org/b/id/8778528-L.jpg
...,...,...,...,...,...,...,...,...,...,...
9780700614936,47687,Peace Pact: The Lost World of the American Fou...,David C. Hendrickson,3.30,0700614931,eng,402,21,0,None
9781594970993,47692,American Gods,Neil Gaiman,4.11,1594970998,spa,477,58,6,None
9780060587017,47693,The Day I Swapped My Dad for Two Goldfish,Neil Gaiman-Dave McKean,4.03,0060587016,en-US,64,8744,524,None
